In [1]:
import os
import time
import requests
import json
import undetected_chromedriver as uc
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from PIL import Image
import pytesseract

# Set the working directory and environment variables
os.chdir('/allah/data')
os.environ['DISPLAY'] = ':10.0'  # Adjust if necessary


# Configure Chrome options
options = Options()
# options.add_argument("start-maximized")  # Open in maximized mode
options.add_argument('--disable-blink-features=AutomationControlled')  # Disable automation control features
options.add_argument('--disable-dev-shm-usage')  # Overcome limited resource problems
options.add_argument('--no-sandbox')  # Bypass OS security model
options.add_argument('--disable-gpu')  # Disable GPU hardware acceleration
options.binary_location = "/usr/bin/google-chrome" 

driver = uc.Chrome(options=options)

highlighted_elements = []

def highlight(element):
    driver.execute_script("arguments[0].style.border='3px solid red'", element)
    highlighted_elements.append(element)

def clear_all_highlights():
    for element in highlighted_elements:
        driver.execute_script("arguments[0].style.border=''", element)
    highlighted_elements.clear()



In [ ]:
driver.get('https://bclub.tk/login/')

# Fill in username and password
username_input = driver.find_element(By.ID, 'id_username')
highlight(username_input)
username_input.send_keys('Brian9977km1')

password_input = driver.find_element(By.ID, 'password')
highlight(password_input)
password_input.send_keys('nimasile')

In [3]:
# Define the base URL and target data path
base_url = 'https://data.binance.vision/?prefix=data/futures/um/daily/'
data_path = '/allah/data'
ticker = 'ETHUSDT'

# Navigate to the page and wait for elements to load
driver.get(base_url)
WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.CSS_SELECTOR, '#listing > tr:nth-child(2) > td:nth-child(1)')))

# Extract links and categories
tbody_element = driver.find_element(By.ID, 'listing')
anchor_elements = tbody_element.find_elements(By.TAG_NAME, 'a')
links = [(anchor.get_attribute('href'), anchor.text.strip('/')) for anchor in anchor_elements][1:]

# Create directories for each category and collect category names
categories = [link[1] for link in links]
for category in categories:
    os.makedirs(os.path.join(data_path, ticker, category), exist_ok=True)


In [6]:
from tqdm.notebook import tqdm
import requests
import os

def download_file(url, download_path):
    response = requests.get(url, stream=True)
    block_size = 1024  # 1 Kibibyte
    with open(download_path, 'wb') as file:
        for data in response.iter_content(block_size):
            file.write(data)

categories = [link[1] for link in links]
kline_categories = [category for category in categories if 'klines' in category]
for category in categories:
    url = f'https://data.binance.vision/?prefix=data/futures/um/daily/{category}/{ticker}/'
    print(url)

    driver.get(url)
    WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.CSS_SELECTOR, '#listing > tr:nth-child(2) > td:nth-child(1)')))

    tbody_element = driver.find_element(By.ID, 'listing')
    anchor_elements = tbody_element.find_elements(By.TAG_NAME, 'a')
    download_links = [(anchor.get_attribute('href'), anchor.text.strip('/')) for anchor in anchor_elements][1:]

    total_files = len(download_links)
    progress_bar = tqdm(total=total_files, unit='file', desc=f'Downloading {category}', colour='green', leave=True, ncols=100, dynamic_ncols=True, position=0)

    for download_link in download_links:
        download_path = os.path.join(data_path, ticker, category, download_link[1])
        if os.path.exists(download_path):
            # print(f'Skipped {download_link[1]} (already exists)')
            progress_bar.update(1)
            continue

        retry_count = 0
        max_retries = 5
        while retry_count < max_retries:
            try:
                download_file(download_link[0], download_path)
                print(f'Downloaded {download_link[1]}')
                progress_bar.update(1)  # Update the progress bar for each downloaded file
                break
            except requests.exceptions.ConnectionError as e:
                retry_count += 1
                print(f'Error downloading {download_link[1]}: {e}. Retrying ({retry_count}/{max_retries})...')
                time.sleep(20)  # Wait for 20 seconds before retrying
        else:
            print(f'Failed to download {download_link[1]} after {max_retries} attempts')

    # Close the progress bar after all downloads in the category are complete
    progress_bar.close()


https://data.binance.vision/?prefix=data/futures/um/daily/aggTrades/ETHUSDT/


KeyboardInterrupt: 

In [8]:
# Extract and process kline categories separately
kline_categories = [category for category in categories if 'klines' in category.lower()]

for kline_category in kline_categories:
    # Construct the URL for kline category
    category_url = f'https://data.binance.vision/?prefix=data/futures/um/daily/{kline_category}/{ticker}/'
    driver.get(category_url)
    WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.CSS_SELECTOR, '#listing > tr:nth-child(2) > td:nth-child(1)')))
    
    # Find and extract time frame links
    tbody_element = driver.find_element(By.ID, 'listing')
    time_frame_links = [(a.get_attribute('href'), a.text.strip('/')) for a in tbody_element.find_elements(By.TAG_NAME, 'a')][1:]
    
    for time_link, time_frame in time_frame_links:
        driver.get(time_link)
        WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.CSS_SELECTOR, '#listing > tr:nth-child(2) > td:nth-child(1)')))
        
        # Extract download links
        tbody_element = driver.find_element(By.ID, 'listing')
        download_links = [(a.get_attribute('href'), a.text.strip('/')) for a in tbody_element.find_elements(By.TAG_NAME, 'a')][1:]
        
        # Process each download link
        for download_link, filename in download_links:
            download_path = os.path.join(data_path, ticker, kline_category, time_frame, filename)
            # make dir if not exist
            os.makedirs(os.path.dirname(download_path), exist_ok=True)
            if os.path.exists(download_path):
                print(f'Skipped {filename} (already exists)')
                continue
            
            # Download file with retry logic
            retry_count = 0
            max_retries = 5
            while retry_count < max_retries:
                try:
                    download_file(download_link, download_path)
                    print(f'Downloaded {filename}')
                    break
                except requests.exceptions.ConnectionError as e:
                    retry_count += 1
                    print(f'Error downloading {filename}: {e}. Retrying ({retry_count}/{max_retries})...')
                    time.sleep(20)
            else:
                print(f'Failed to download {filename} after {max_retries} attempts')


Downloaded ETHUSDT-12h-2024-07-22.zip.CHECKSUM
Downloaded ETHUSDT-12h-2024-07-22.zip
Downloaded ETHUSDT-12h-2024-07-21.zip.CHECKSUM
Downloaded ETHUSDT-12h-2024-07-21.zip
Downloaded ETHUSDT-12h-2024-07-20.zip.CHECKSUM
Downloaded ETHUSDT-12h-2024-07-20.zip
Downloaded ETHUSDT-12h-2024-07-19.zip.CHECKSUM
Downloaded ETHUSDT-12h-2024-07-19.zip
Downloaded ETHUSDT-12h-2024-07-18.zip.CHECKSUM
Downloaded ETHUSDT-12h-2024-07-18.zip
Downloaded ETHUSDT-12h-2024-07-17.zip.CHECKSUM
Downloaded ETHUSDT-12h-2024-07-17.zip
Downloaded ETHUSDT-12h-2024-07-16.zip.CHECKSUM
Downloaded ETHUSDT-12h-2024-07-16.zip
Downloaded ETHUSDT-12h-2024-07-15.zip.CHECKSUM
Downloaded ETHUSDT-12h-2024-07-15.zip
Downloaded ETHUSDT-12h-2024-07-14.zip.CHECKSUM
Downloaded ETHUSDT-12h-2024-07-14.zip
Downloaded ETHUSDT-12h-2024-07-13.zip.CHECKSUM
Downloaded ETHUSDT-12h-2024-07-13.zip
Downloaded ETHUSDT-12h-2024-07-12.zip.CHECKSUM
Downloaded ETHUSDT-12h-2024-07-12.zip
Downloaded ETHUSDT-12h-2024-07-11.zip.CHECKSUM
Downloaded ETHUSDT

KeyboardInterrupt: 

In [9]:
from tqdm.notebook import tqdm
import requests
import os
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

def download_file(url, download_path):
    response = requests.get(url, stream=True)
    with open(download_path, 'wb') as file:
        for data in response.iter_content(1024):
            file.write(data)

def process_category(category, ticker, base_url):
    url = f'{base_url}{category}/{ticker}/'
    driver.get(url)
    WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.CSS_SELECTOR, '#listing > tr:nth-child(2) > td:nth-child(1)')))
    
    tbody_element = driver.find_element(By.ID, 'listing')
    download_links = [(a.get_attribute('href'), a.text.strip('/')) for a in tbody_element.find_elements(By.TAG_NAME, 'a')][1:]
    
    total_files = len(download_links)
    progress_bar = tqdm(total=total_files, unit='file', desc=f'Downloading {category}', colour='green', leave=True, ncols=100, dynamic_ncols=True, position=0)
    
    for link, filename in download_links:
        download_path = os.path.join(data_path, ticker, category, filename)
        os.makedirs(os.path.dirname(download_path), exist_ok=True)
        if os.path.exists(download_path):
            print(f'Skipped {filename} (already exists)')
            progress_bar.update(1)
            continue
        
        retry_count = 0
        while retry_count < 5:
            try:
                download_file(link, download_path)
                progress_bar.update(1)
                break
            except requests.exceptions.ConnectionError as e:
                retry_count += 1
                print(f'Error downloading {filename}: {e}. Retrying ({retry_count}/5)...')
                time.sleep(20)
        else:
            print(f'Failed to download {filename} after 5 attempts')
    
    progress_bar.close()

def process_kline_category(kline_category, ticker, base_url):
    category_url = f'{base_url}{kline_category}/{ticker}/'
    driver.get(category_url)
    WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.CSS_SELECTOR, '#listing > tr:nth-child(2) > td:nth-child(1)')))
    
    tbody_element = driver.find_element(By.ID, 'listing')
    time_frame_links = [(a.get_attribute('href'), a.text.strip('/')) for a in tbody_element.find_elements(By.TAG_NAME, 'a')][1:]
    
    for time_link, time_frame in time_frame_links:
        driver.get(time_link)
        WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.CSS_SELECTOR, '#listing > tr:nth-child(2) > td:nth-child(1)')))
        
        tbody_element = driver.find_element(By.ID, 'listing')
        download_links = [(a.get_attribute('href'), a.text.strip('/')) for a in tbody_element.find_elements(By.TAG_NAME, 'a')][1:]
        
        total_files = len(download_links)
        progress_bar = tqdm(total=total_files, unit='file', desc=f'Downloading {kline_category} - {time_frame}', colour='green', leave=True, ncols=100, dynamic_ncols=True, position=0)
        
        for link, filename in download_links:
            download_path = os.path.join(data_path, ticker, kline_category, time_frame, filename)
            os.makedirs(os.path.dirname(download_path), exist_ok=True)
            if os.path.exists(download_path):
                print(f'Skipped {filename} (already exists)')
                progress_bar.update(1)
                continue
            
            retry_count = 0
            while retry_count < 5:
                try:
                    download_file(link, download_path)
                    progress_bar.update(1)
                    break
                except requests.exceptions.ConnectionError as e:
                    retry_count += 1
                    print(f'Error downloading {filename}: {e}. Retrying ({retry_count}/5)...')
                    time.sleep(20)
            else:
                print(f'Failed to download {filename} after 5 attempts')
        
        progress_bar.close()

# Define the base URL and target data path
base_url = 'https://data.binance.vision/?prefix=data/futures/um/daily/'
data_path = '/allah/data'
ticker = 'ETHUSDT'

# Extract categories
categories = [link[1] for link in links]
kline_categories = [category for category in categories if 'klines' in category.lower()]

# Process all categories
for category in categories:
    if category in kline_categories:
        process_kline_category(category, ticker, base_url)
    else:
        process_category(category, ticker, base_url)


Skipped ETHUSDT-aggTrades-2024-07-22.zip.CHECKSUM (already exists)
Skipped ETHUSDT-aggTrades-2024-07-22.zip (already exists)
Skipped ETHUSDT-aggTrades-2024-07-21.zip.CHECKSUM (already exists)
Skipped ETHUSDT-aggTrades-2024-07-21.zip (already exists)
Skipped ETHUSDT-aggTrades-2024-07-20.zip.CHECKSUM (already exists)
Skipped ETHUSDT-aggTrades-2024-07-20.zip (already exists)
Skipped ETHUSDT-aggTrades-2024-07-19.zip.CHECKSUM (already exists)
Skipped ETHUSDT-aggTrades-2024-07-19.zip (already exists)
Skipped ETHUSDT-aggTrades-2024-07-18.zip.CHECKSUM (already exists)
Skipped ETHUSDT-aggTrades-2024-07-18.zip (already exists)
Skipped ETHUSDT-aggTrades-2024-07-17.zip.CHECKSUM (already exists)
Skipped ETHUSDT-aggTrades-2024-07-17.zip (already exists)
Skipped ETHUSDT-aggTrades-2024-07-16.zip.CHECKSUM (already exists)
Skipped ETHUSDT-aggTrades-2024-07-16.zip (already exists)
Skipped ETHUSDT-aggTrades-2024-07-15.zip.CHECKSUM (already exists)
Skipped ETHUSDT-aggTrades-2024-07-15.zip (already exists)


Skipped ETHUSDT-bookDepth-2024-07-22.zip.CHECKSUM (already exists)
Skipped ETHUSDT-bookDepth-2024-07-22.zip (already exists)
Skipped ETHUSDT-bookDepth-2024-07-21.zip.CHECKSUM (already exists)
Skipped ETHUSDT-bookDepth-2024-07-21.zip (already exists)
Skipped ETHUSDT-bookDepth-2024-07-20.zip.CHECKSUM (already exists)
Skipped ETHUSDT-bookDepth-2024-07-20.zip (already exists)
Skipped ETHUSDT-bookDepth-2024-07-19.zip.CHECKSUM (already exists)
Skipped ETHUSDT-bookDepth-2024-07-19.zip (already exists)
Skipped ETHUSDT-bookDepth-2024-07-18.zip.CHECKSUM (already exists)
Skipped ETHUSDT-bookDepth-2024-07-18.zip (already exists)
Skipped ETHUSDT-bookDepth-2024-07-17.zip.CHECKSUM (already exists)
Skipped ETHUSDT-bookDepth-2024-07-17.zip (already exists)
Skipped ETHUSDT-bookDepth-2024-07-16.zip.CHECKSUM (already exists)
Skipped ETHUSDT-bookDepth-2024-07-16.zip (already exists)
Skipped ETHUSDT-bookDepth-2024-07-15.zip.CHECKSUM (already exists)
Skipped ETHUSDT-bookDepth-2024-07-15.zip (already exists)


Skipped ETHUSDT-bookTicker-2024-03-30.zip.CHECKSUM (already exists)
Skipped ETHUSDT-bookTicker-2024-03-30.zip (already exists)
Skipped ETHUSDT-bookTicker-2024-03-29.zip.CHECKSUM (already exists)
Skipped ETHUSDT-bookTicker-2024-03-29.zip (already exists)
Skipped ETHUSDT-bookTicker-2024-03-28.zip.CHECKSUM (already exists)
Skipped ETHUSDT-bookTicker-2024-03-28.zip (already exists)
Skipped ETHUSDT-bookTicker-2024-03-27.zip.CHECKSUM (already exists)
Skipped ETHUSDT-bookTicker-2024-03-27.zip (already exists)
Skipped ETHUSDT-bookTicker-2024-03-26.zip.CHECKSUM (already exists)
Skipped ETHUSDT-bookTicker-2024-03-26.zip (already exists)
Skipped ETHUSDT-bookTicker-2024-03-25.zip.CHECKSUM (already exists)
Skipped ETHUSDT-bookTicker-2024-03-25.zip (already exists)
Skipped ETHUSDT-bookTicker-2024-03-24.zip.CHECKSUM (already exists)
Skipped ETHUSDT-bookTicker-2024-03-24.zip (already exists)
Skipped ETHUSDT-bookTicker-2024-03-23.zip.CHECKSUM (already exists)
Skipped ETHUSDT-bookTicker-2024-03-23.zip (

Skipped ETHUSDT-12h-2024-07-22.zip.CHECKSUM (already exists)
Skipped ETHUSDT-12h-2024-07-22.zip (already exists)
Skipped ETHUSDT-12h-2024-07-21.zip.CHECKSUM (already exists)
Skipped ETHUSDT-12h-2024-07-21.zip (already exists)
Skipped ETHUSDT-12h-2024-07-20.zip.CHECKSUM (already exists)
Skipped ETHUSDT-12h-2024-07-20.zip (already exists)
Skipped ETHUSDT-12h-2024-07-19.zip.CHECKSUM (already exists)
Skipped ETHUSDT-12h-2024-07-19.zip (already exists)
Skipped ETHUSDT-12h-2024-07-18.zip.CHECKSUM (already exists)
Skipped ETHUSDT-12h-2024-07-18.zip (already exists)
Skipped ETHUSDT-12h-2024-07-17.zip.CHECKSUM (already exists)
Skipped ETHUSDT-12h-2024-07-17.zip (already exists)
Skipped ETHUSDT-12h-2024-07-16.zip.CHECKSUM (already exists)
Skipped ETHUSDT-12h-2024-07-16.zip (already exists)
Skipped ETHUSDT-12h-2024-07-15.zip.CHECKSUM (already exists)
Skipped ETHUSDT-12h-2024-07-15.zip (already exists)
Skipped ETHUSDT-12h-2024-07-14.zip.CHECKSUM (already exists)
Skipped ETHUSDT-12h-2024-07-14.zip 